# 05 - Connect to Neo4j and Use Database `hpo`

## Learning objectives
- Connect to a Neo4j instance from Python using the official driver.
- Create constraints, indexes, and initialize n10s configuration (only if the graph is empty).
- Run a simple read query on database `hpo`.
- Understand where credentials should live in production.

In [39]:
from neo4j import GraphDatabase

In [40]:
# Connection settings.
# In production, keep credentials in a .env file or a secrets manager, not in source code.
uri = "neo4j://127.0.0.1:7687"
user = "neo4j"
password = "12345678"
database = "hpo"

driver = GraphDatabase.driver(uri, auth=(user, password))
driver.verify_connectivity()
print(f"Connected to Neo4j at {uri}")

Connected to Neo4j at neo4j://127.0.0.1:7687


## Step 1 - Create Constraints and Indexes

Before loading RDF data at scale, we define schema rules that keep the graph consistent and faster to query:

- **Uniqueness constraints on `:Resource(uri)` and `:Resource(id)`** prevent duplicate Resource nodes for the same real-world entity.
- **Indexes on `:HpoDisease(id)` and `:HpoPhenotype(id)`** speed up lookup operations during KG construction and retrieval queries.
- In this course model, **`HpoDisease`** represents disease nodes and **`HpoPhenotype`** represents phenotypic abnormality nodes.

This is a standard preparation step before import: cleaner graph, safer merges, and better query performance.

In [41]:
# Ensure the target database exists (requires Enterprise or Neo4j Desktop).
with driver.session(database="system") as session:
    session.run(f"CREATE DATABASE {database} IF NOT EXISTS").consume()
print(f"Database '{database}' is ready.")

# Check whether the graph already contains data.
# n10s graphconfig.init() cannot be run (and config cannot be changed) once data exists.
with driver.session(database=database) as session:
    node_count = session.run("MATCH (n) RETURN count(n) AS c").single()["c"]

graph_is_empty = node_count == 0
print(f"Current node count: {node_count:,}")
if graph_is_empty:
    print("Graph is EMPTY → constraints, indexes, and n10s config will be initialized.")
else:
    print("Graph is already POPULATED → skipping schema and n10s initialization (config cannot be changed).")

Database 'hpo' is ready.
Current node count: 0
Graph is EMPTY → constraints, indexes, and n10s config will be initialized.


In [42]:
# Create constraints and indexes for n10s/HPO data.
# Only needed when the graph is empty; once data exists, these should already be in place.
if graph_is_empty:
    schema_statements = [
        "CREATE CONSTRAINT n10s_unique_uri IF NOT EXISTS FOR (r:Resource) REQUIRE r.uri IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Resource) REQUIRE (n.id) IS UNIQUE",
        "CREATE INDEX disease_id IF NOT EXISTS FOR (n:HpoDisease) ON (n.id)",
        "CREATE INDEX phenotype_id IF NOT EXISTS FOR (n:HpoPhenotype) ON (n.id)",
    ]

    with driver.session(database=database) as session:
        for stmt in schema_statements:
            session.execute_write(lambda tx, s=stmt: tx.run(s).consume())

    print("Constraints and indexes created.")
else:
    print("Skipped — graph already populated, constraints and indexes should already exist.")

Constraints and indexes created.


In [43]:
# Show all constraints and indexes defined in the database.
with driver.session(database=database) as session:
    print("=== Constraints ===")
    constraints = session.run("SHOW CONSTRAINTS").data()
    for c in constraints:
        print(f"  {c['name']}  |  type: {c['type']}  |  entity: {c['entityType']}  |  properties: {c['properties']}")

    print("\n=== Indexes ===")
    indexes = session.run("SHOW INDEXES").data()
    for idx in indexes:
        print(f"  {idx['name']}  |  type: {idx['type']}  |  entity: {idx['entityType']}  |  properties: {idx['properties']}")

=== Constraints ===
  constraint_e35c7fe  |  type: UNIQUENESS  |  entity: NODE  |  properties: ['id']
  n10s_unique_uri  |  type: UNIQUENESS  |  entity: NODE  |  properties: ['uri']

=== Indexes ===
  constraint_e35c7fe  |  type: RANGE  |  entity: NODE  |  properties: ['id']
  disease_id  |  type: RANGE  |  entity: NODE  |  properties: ['id']
  index_343aff4e  |  type: LOOKUP  |  entity: NODE  |  properties: None
  index_f7700477  |  type: LOOKUP  |  entity: RELATIONSHIP  |  properties: None
  n10s_unique_uri  |  type: RANGE  |  entity: NODE  |  properties: ['uri']
  phenotype_id  |  type: RANGE  |  entity: NODE  |  properties: ['id']


## Step 2 - Initialize Neosemantics Configuration

After creating constraints and indexes, we define the initial `n10s` configuration used during RDF import.

- `handleVocabUris: "IGNORE"` ignores namespace prefixes in imported terms, producing shorter graph names.
- `applyNeo4jNaming: true` applies Neo4j naming conventions and standard formatting for relationship types, making queries more consistent.

These settings help normalize ontology-derived data before large-scale KG ingestion.

In [44]:
# Initialize and configure n10s for RDF import.
# n10s.graphconfig.init() can only be called on an empty graph; once data exists the config
# is locked and cannot be changed.
if graph_is_empty:
    with driver.session(database=database) as session:
        # Step 1: Initialize the graph config (creates the _GraphConfig node).
        session.execute_write(lambda tx: tx.run("CALL n10s.graphconfig.init()").consume())
        print("n10s.graphconfig.init() executed.")

        # Step 2: Set import options.
        session.execute_write(lambda tx: tx.run('CALL n10s.graphconfig.set({ handleVocabUris: "IGNORE" })').consume())
        session.execute_write(lambda tx: tx.run('CALL n10s.graphconfig.set({ applyNeo4jNaming: true })').consume())

    print("n10s configuration updated.")
else:
    print("Skipped — graph already populated, n10s config cannot be changed.")

# Show the current n10s config (works regardless of whether the graph is empty or populated).
with driver.session(database=database) as session:
    config = session.run("CALL n10s.graphconfig.show()").data()
    print("\nCurrent n10s configuration:")
    for entry in config:
        print(f"  {entry}")

n10s.graphconfig.init() executed.
n10s configuration updated.

Current n10s configuration:
  {'param': 'handleVocabUris', 'value': 'IGNORE'}
  {'param': 'handleMultival', 'value': 'OVERWRITE'}
  {'param': 'handleRDFTypes', 'value': 'LABELS'}
  {'param': 'keepLangTag', 'value': False}
  {'param': 'keepCustomDataTypes', 'value': False}
  {'param': 'applyNeo4jNaming', 'value': True}
  {'param': 'baseSchemaNamespace', 'value': 'neo4j://graph.schema#'}
  {'param': 'baseSchemaPrefix', 'value': 'n4sch'}
  {'param': 'classLabel', 'value': 'Class'}
  {'param': 'subClassOfRel', 'value': 'SCO'}
  {'param': 'dataTypePropertyLabel', 'value': 'Property'}
  {'param': 'objectPropertyLabel', 'value': 'Relationship'}
  {'param': 'subPropertyOfRel', 'value': 'SPO'}
  {'param': 'domainRel', 'value': 'DOMAIN'}
  {'param': 'rangeRel', 'value': 'RANGE'}
  {'param': 'classNamePropName', 'value': 'name'}
  {'param': 'relNamePropName', 'value': 'name'}


In [45]:
# Optional: count nodes to confirm access to graph data.
with driver.session(database=database) as session:
    total_nodes = session.run("MATCH (n) RETURN count(n) AS c").single()["c"]

print(f"Total nodes in '{database}': {total_nodes}")

Total nodes in 'hpo': 1


In [46]:
driver.close()
print("Neo4j driver closed.")

Neo4j driver closed.


## Exercises
1. Replace `MATCH (n) RETURN count(n)` with `MATCH (n) RETURN labels(n), count(*)` and inspect label distribution.
2. Write a query that returns 5 nodes and 5 relationships from `hpo`.
3. Move credentials to environment variables and load them with `os.getenv`.